# Besoin Client 1 : Visualisation sur carte

## Contexte

Dans ce projet, nous cherchons à regrouper des arbres selon leur taille à partir de données réelles.

## Objectif

Le but de cette partie est de permettre la visualisation des différents arbres de la base de données selon leur taille à partir d'une carte. Les arbres devront être triés en fonction de s'ils sont grand ou petit, ou petit, moyen ou grand. Dans ce Notebook, nous utiliserons le test de 3 clusters.

## Méthode

Le choix s'est tourné vers la méthode K-Means. Cela est dû au fait que nous n'avons pas de label, nous sommes en apprentissage non-supervisé. On veut également que notre modèle tri regroupe automatiquement selon la taille des arbres. En plus, nous n'avons que des données simple, seulement la taille, qui est parfait pour l'utilisaton de K-Means

### 1 - Programme taille_cluster.py

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.cluster import KMeans
from pyproj import Transformer
import plotly.express as px
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import joblib

# Suppresion des erreurs n'empêchant pas l'exécution
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

## Chargement des données

La base de données doit être chargée afin de récupérer toutes les données des arbres.

In [2]:
csv_path = Path("../../data/Data_Arbre_Clean.csv")
data = pd.read_csv(csv_path)

data.head()

,X,Y,clc_quartier,clc_secteur,id_arbre,haut_tot,haut_tronc,tronc_diam,fk_arb_etat,fk_stadedev,...,fk_revetement,dte_plantation,age_estim,clc_nbr_diag,dte_abattage,villeca,nomfrancais,nomlatin,feuillage,remarquable
0,1.720320e+06,8.294619e+06,Quartier du Centre-Ville,Boulevard Richelieu,24.0,0.0,0.0,0.0,SUPPRIMÉ,NaN,...,NaN,NaN,0.0,0.0,2015-07-01T00:00:00Z,VILLE,NaN,NaN,NaN,False
1,1.720898e+06,8.293531e+06,Quartier du Centre-Ville,Boulevard Léon Blum,24.0,0.0,0.0,0.0,ABATTU,NaN,...,NaN,NaN,0.0,0.0,NaN,VILLE,NaN,NaN,NaN,False
2,1.720894e+06,8.293542e+06,Quartier du Centre-Ville,Boulevard Léon Blum,53.0,0.0,0.0,0.0,SUPPRIMÉ,NaN,...,NaN,NaN,0.0,0.0,NaN,VILLE,NaN,NaN,NaN,False
3,1.720902e+06,8.293545e+06,Quartier du Centre-Ville,Boulevard Léon Blum,54.0,0.0,0.0,0.0,SUPPRIMÉ,NaN,...,NaN,NaN,0.0,0.0,NaN,VILLE,NaN,NaN,NaN,False
4,1.721089e+06,8.293619e+06,Quartier du Centre-Ville,Boulevard Léon Blum,63.0,0.0,0.0,0.0,ABATTU,NaN,...,NaN,NaN,0.0,0.0,NaN,VILLE,NaN,NaN,NaN,False


## Préparation des données

On sélectionne les données utiles : 
    - la taille de l'arbre
    - ses coordonnées géographiques

Nous supprimons également tous les arbres ayant aucune donnée traitable ( NA ou 0) dans ces colonnes.

In [3]:
data = data[['haut_tot', 'X', 'Y']]

data = data.dropna()
data = data[data['haut_tot'] > 0]

data.describe()

,haut_tot,X,Y
count,10265.000000,1.026500e+04,1.026500e+04
mean,10.766001,1.720827e+06,8.294591e+06
std,5.810493,1.318115e+03,1.309974e+03
min,1.000000,1.717296e+06,8.290361e+06
25%,6.000000,1.719775e+06,8.293706e+06
50%,9.000000,1.721205e+06,8.294820e+06
75%,15.000000,1.721784e+06,8.295568e+06
max,37.000000,1.723497e+06,8.298783e+06


## Clustering avec KMeans

Le clustering consiste à regrouper les données en fonction de leur similarité.

Ici, on utilise simplement la hauteur des arbres pour les traiter.
On mets par défaut un cluster à 3, mais dans le programme Python, l'utilisateur reçoit une demande dans le terminal afin de choisir s'il veut 2 ou 3 clusters.

In [4]:
X = data[['haut_tot']]


''' CODE INITIALE DU FICHIER PYTHON
while True: 
    try:
        k = int(input("\nChoisir le nombre de clusters (2 ou 3) : "))
        if k in [2, 3]:
            break
        else:
            print("Veuillez entrer 2 ou 3.")
    except ValueError:
        print("Veuillez entrer un nombre valide.")
'''

k = 3  # à tester : 2 ou 3

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
data['cluster'] = kmeans.fit_predict(X)

data.head()

,haut_tot,X,Y,cluster
9,6.0,1.721096e+06,8.293515e+06,1
20,13.0,1.719657e+06,8.295835e+06,0
21,12.0,1.720792e+06,8.293960e+06,0
24,16.0,1.721742e+06,8.295884e+06,0
25,5.0,1.721863e+06,8.295514e+06,1


## Interprétation des clusters

Les clusters générés par KMeans n'ont pas de signification directe.  
On les ordonnes donc en fonction de leur hauteur moyenne.

In [5]:
cluster_means = data.groupby('cluster')['haut_tot'].mean()
sorted_clusters = cluster_means.sort_values().index.tolist()

mapping = {old: new for new, old in enumerate(sorted_clusters)}

''' SAUVEGARDE POUR RÉFÉRENCE DU FICHIER PYTHON
joblib.dump(mapping, "mapping_clusters.pkl")
'''
data['cluster'] = data['cluster'].map(mapping)

if k == 2:
    labels = {0: "Petit", 1: "Grand"}
elif k == 3:
    labels = {0: "Petit", 1: "Moyen", 2: "Grand"}
data['type_arbre'] = data['cluster'].map(labels)

data[['haut_tot', 'cluster', 'type_arbre']].head()

,haut_tot,cluster,type_arbre
9,6.0,0,Petit
20,13.0,1,Moyen
21,12.0,1,Moyen
24,16.0,1,Moyen
25,5.0,0,Petit


## Évaluation du clustering

Nous utilisons trois métriques pour évaluer la qualité du clustering :

- **Silhouette Coefficient** : mesure la cohérence des clusters (proximité des points au sein d’un même groupe et éloignement des autres groupes).  
  -> Sa valeur est comprise entre -1 et 1.  
  -> Une valeur proche de 1 indique un bon clustering.

- **Calinski-Harabasz Index** : mesure la séparation entre les clusters.  
  -> Plus cet indice est élevé, meilleure est la séparation entre les groupes.

- **Davies-Bouldin Index** : mesure le chevauchement entre les clusters.  
  -> Plus cet indice est faible, meilleure est la séparation.

### Interprétation des résultats

Les résultats obtenus montrent que :

- Le **Silhouette Score** indique si les arbres sont bien regroupés selon leur taille.
- Le **Calinski-Harabasz Index** confirme si les groupes sont bien distincts les uns des autres.
- Le **Davies-Bouldin Index** permet de vérifier que les clusters ne se chevauchent pas.

Dans notre cas, les valeurs obtenues suggèrent que le clustering est cohérent.Cela permet une séparation des arbres en différentes catégories de taille (petit, moyen, grand).

In [6]:
silhouette = silhouette_score(X, data['cluster'])
calinski = calinski_harabasz_score(X, data['cluster'])
davies = davies_bouldin_score(X, data['cluster'])

print("Silhouette :", silhouette)
print("Calinski-Harabasz :", calinski)
print("Davies-Bouldin :", davies)

Silhouette : 0.6294762089042186
Calinski-Harabasz : 30538.50130751263
Davies-Bouldin : 0.5060505180155337


## Conversion des coordonnées

Les coordonnées initiales sont converties en coordonnées GPS afin de permettre une visualisation sur OpenStreetMap.

In [7]:
transformer = Transformer.from_crs("EPSG:3949", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(data["X"].values, data["Y"].values)
data["lon"] = lon
data["lat"] = lat

## Visualisation sur carte

Les arbres sont affichés sur une carte interactive, avec une couleur correspondant à leur catégorie.

In [8]:
fig = px.scatter_map(
    data,
    lat="lat",
    lon="lon",
    color="type_arbre",
    text="type_arbre",
    zoom=12,
    opacity=0.6,
    title="Carte des arbres par catégorie"
)

fig.update_layout(map_style="open-street-map")

## Analyse descriptive des clusters

Ensuite, nous réalisons un affichage des clusters afin de pouvoir les analyser.

In [9]:
stats = data.groupby('cluster')['haut_tot'].agg(['count', 'min', 'max', 'mean'])

stats.columns = ['Nombre_arbres', 'Minimum', 'Maximum', 'Moyenne']
stats['Type_arbre'] = stats.index.map(labels)

stats

,Nombre_arbres,Minimum,Maximum,Moyenne,Type_arbre
cluster,,,,,
0,6000,1.0,10.0,6.710500,Petit
1,3140,11.0,18.0,14.241720,Moyen
2,1125,19.0,37.0,22.694222,Grand
